## tl;dr

This notebook aggregates the retained canonical HSR values by team and season. It calculates HSR share only on rows where both canonical HSR and the existing `distance_total_m_clean` value are present and numeric.

## Context & Methods

### Key Assumptions

The denominator uses the existing accepted total-distance field without replacement from raw sources. Missing HSR is not treated as zero. Team-specific units, thresholds and calibration remain as documented in the mapping manifest.

## Data

In [ ]:
import csv
import json
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd

root = Path.cwd().resolve()
while root != root.parent and not (root / 'docs/evidence/hsr_exposure_mapping_manifest.json').exists():
    root = root.parent
manifest = json.loads((root / 'docs/evidence/hsr_exposure_mapping_manifest.json').read_text())
metadata = {(row['season'], row['team_key']): row for row in manifest['team_seasons']}
output_files = sorted((root / 'data/hsr_exposure').glob('*/*/exposure_with_hsr.csv'))
assert len(output_files) == 32

## Results

In [ ]:
results = []
for path in output_files:
    season, team = path.parts[-3:-1]
    hsr_sum = Decimal(0)
    paired_hsr_sum = Decimal(0)
    paired_total_sum = Decimal(0)
    accepted_rows = hsr_rows = paired_rows = 0
    with path.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            accepted_rows += 1
            hsr_text = row.get('high speed running distance', '').strip()
            total_text = row.get('distance_total_m_clean', '').strip()
            if not hsr_text:
                continue
            hsr_value = Decimal(hsr_text)
            hsr_sum += hsr_value
            hsr_rows += 1
            if not total_text:
                continue
            try:
                total_value = Decimal(total_text)
            except InvalidOperation:
                continue
            paired_hsr_sum += hsr_value
            paired_total_sum += total_value
            paired_rows += 1
    source_available = metadata[(season, team)]['source_available']
    hsr_share = paired_hsr_sum / paired_total_sum * 100 if source_available and paired_total_sum else None
    results.append({
        'team': team.title(),
        'season': season,
        'aggregated_hsr': float(hsr_sum) if source_available else None,
        'paired_total_distance': float(paired_total_sum) if source_available else None,
        'hsr_share_percent': float(hsr_share) if hsr_share is not None else None,
        'hsr_rows': hsr_rows,
        'paired_rows': paired_rows,
        'accepted_rows': accepted_rows,
        'unit_status': metadata[(season, team)]['units'],
    })

summary = pd.DataFrame(results).sort_values(['season', 'team']).reset_index(drop=True)
assert len(summary) == 32
assert summary['hsr_share_percent'].dropna().between(0, 100).all()
summary

## Takeaways

The percentage is a paired-row ratio, not a claim that blank HSR values are zero. Benetton and Edinburgh have no valid 2025-26 HSR source. Zebre 2025-26 retains implausibly large accepted total-distance values, so its percentage is mechanically reproducible but not analytically credible until those accepted totals are corrected through the governed pipeline.